# Parquet 데이터 탐색 with DuckDB

파일 경로만 바꾸고 위에서부터 실행하세요. 특정 도메인이나 컬럼명을 가정하지 않습니다.

1. 파일과 스키마 확인
2. 미리보기와 자료형별 프로파일
3. 값 빈도와 숫자 분포
4. 조건 조회, 정렬, 페이지 이동
5. STRUCT / LIST / ARRAY / MAP 탐색
6. 직접 SQL과 결과 내보내기

먼저 `uv sync`를 실행하고 커널을 `.venv/bin/python`으로 선택합니다.
기본 프로파일은 **앞 10,000행** 기준이며 무작위 표본이 아닙니다. 전체 통계는 `PROFILE_ROWS = None`으로 선택합니다.
필터를 바꾸면 해당 설정 셀과 그 아래 실행 셀을 다시 실행하세요. 파일을 바꾸면 전체 셀을 다시 실행하세요.

In [3]:
from pathlib import Path
from glob import glob
from numbers import Integral

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 50)

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "DuckDB_notebook"]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "parquet_query.ipynb").is_file()), Path.cwd()
)
print(f"기준 폴더: {PROJECT_ROOT}")
print(f"DuckDB {duckdb.__version__}")

기준 폴더: /home/bjeong/GitHub/DuckDB_notebook
DuckDB 1.5.6


## 1. 파일 열기

`PARQUET_PATHS`에는 파일 하나, glob 패턴, 폴더, 또는 이들을 담은 목록을 입력합니다.
폴더는 하위의 `*.parquet` 파일까지 찾습니다. 상대 경로는 위 기준 폴더 기준입니다.

```python
PARQUET_PATHS = "data/example.parquet"
PARQUET_PATHS = "data/**/*.parquet"
PARQUET_PATHS = ["data/part-01.parquet", "data/part-02.parquet"]
```

여러 파일의 컬럼은 이름으로 합칩니다. 없는 컬럼은 NULL로 채워지지만 호환되지 않는 타입은 오류가 날 수 있습니다.
서로 무관한 데이터는 파일을 따로 선택하세요. Hive 디렉터리(`year=2026/`)는 기본 자동 감지합니다.
WSL에서는 Windows 파일 경로를 `/mnt/c/...`처럼 입력하세요. 이 노트북의 파일 선택기는 로컬 파일용입니다.

In [ ]:
PARQUET_PATHS = "samples/*.parquet"
UNION_BY_NAME = True
HIVE_PARTITIONING = None  # None: 자동 감지, True/False: 명시적으로 지정
TIMEZONE = "UTC"  # TIMESTAMPTZ 표시 및 시간대 없는 입력 해석


def resolve_files(paths):
    patterns = [paths] if isinstance(paths, (str, Path)) else list(paths)
    if not patterns:
        raise ValueError("파일 경로를 하나 이상 입력하세요.")
    files = set()
    for pattern in patterns:
        path = Path(pattern).expanduser()
        path = path if path.is_absolute() else PROJECT_ROOT / path
        if path.is_dir():
            matches = list(path.rglob("*.parquet"))
        elif path.is_file():
            matches = [path]  # 파일명에 [ 또는 ]가 있어도 그대로 읽습니다.
        else:
            matches = [Path(p) for p in glob(str(path), recursive=True)]
        matches = [p.resolve() for p in matches if p.is_file()]
        if not matches:
            raise FileNotFoundError(f"일치하는 파일이 없습니다: {path}")
        files.update(matches)
    return sorted(str(p) for p in files)


files = resolve_files(PARQUET_PATHS)
if "con" in globals():
    con.close()
con = duckdb.connect(":memory:")
result = nested_result = sql_result = None
result_relations = {}
con.execute("SET TimeZone = ?", [TIMEZONE])
options = {"union_by_name": UNION_BY_NAME}
if HIVE_PARTITIONING is not None:
    options["hive_partitioning"] = HIVE_PARTITIONING
con.read_parquet(files, **options).create_view("data")
file_info = con.execute("SELECT * FROM parquet_file_metadata(?)", [files]).fetchdf()
print(f"파일 {len(files):,}개 / 메타데이터상 {int(file_info['num_rows'].sum()):,}행")
display(file_info)

## 2. 스키마와 미리보기

`schema`는 DuckDB가 해석한 최상위 컬럼, `field_catalog`는 STRUCT 내부까지 내려간 필드 목록입니다.
중첩 필드는 `("상위", "하위")` 경로로 선택합니다. 점이 포함된 실제 컬럼명은 문자열 그대로 지정하면 됩니다.
LIST/ARRAY의 요소와 MAP의 key/value는 뒤의 중첩 탐색에서 펼칩니다.

미리보기는 저장 순서상 앞부분입니다. 별도 정렬 전에는 행 순서를 보장하지 않습니다.
중첩값은 화면에서 축약되지만 `preview`에는 원래 값이 들어 있습니다.

In [ ]:
PREVIEW_ROWS = 10
SHOW_PHYSICAL_SCHEMA = False


def positive_int(value, name, allow_zero=False):
    minimum = 0 if allow_zero else 1
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{name}은 {minimum} 이상의 정수여야 합니다.")
    return int(value)


def qi(name):
    return '"' + name.replace('"', '""') + '"'


relation = con.table("data")
root_types = dict(zip(relation.columns, relation.types))
field_types = {}


def walk_fields(path, dtype):
    field_types[path] = dtype
    if dtype.id == "struct":
        for name, child_type in dtype.children:
            walk_fields((*path, name), child_type)


for name, dtype in root_types.items():
    walk_fields((name,), dtype)


def field_path(field):
    path = (field,) if isinstance(field, str) else tuple(field)
    if path not in field_types:
        raise ValueError(f"없는 필드: {path}. field_catalog를 확인하세요.")
    return path


def field_sql(field):
    return ".".join(qi(part) for part in field_path(field))


COMPLEX_TYPES = {"struct", "list", "array", "map", "union"}
NUMERIC_TYPES = {
    "tinyint", "smallint", "integer", "bigint", "hugeint", "bignum",
    "utinyint", "usmallint", "uinteger", "ubigint", "uhugeint",
    "float", "double", "decimal",
}
schema = con.execute("DESCRIBE data").fetchdf()
field_catalog = pd.DataFrame([
    {"path": path, "type": str(dtype), "kind": dtype.id}
    for path, dtype in field_types.items()
])
display(schema)
display(field_catalog)
preview = con.table("data").limit(positive_int(PREVIEW_ROWS, "PREVIEW_ROWS")).df()
display(preview)
if SHOW_PHYSICAL_SCHEMA:
    display(con.execute("SELECT * FROM parquet_schema(?)", [files]).fetchdf())

## 3. 결측값, 고유값 수, 범위

프로파일은 최상위 컬럼을 대상으로 합니다. 고유값 수는 **근사치**, NULL 개수는 선택한 분석 범위의 정확한 값입니다.
STRUCT 내부의 NULL과 LIST 안의 NULL은 부모 컬럼의 NULL과 다릅니다.
중첩 컬럼의 고유값/최솟값/최댓값은 계산하지 않습니다. 범위값은 비교를 끝낸 뒤 문자열로 표시합니다.

`PROFILE_ROWS = None`은 전체 파일을 스캔할 수 있습니다. 행 수 제한도 단일 행의 매우 큰 중첩값까지 제한하지는 않습니다.

In [ ]:
PROFILE_ROWS = 10_000  # None이면 전체 데이터

profile_limit = "" if PROFILE_ROWS is None else f" LIMIT {positive_int(PROFILE_ROWS, 'PROFILE_ROWS')}"
con.execute(f"CREATE OR REPLACE TEMP VIEW profile_data AS SELECT * FROM data{profile_limit}")
profile_count = con.execute("SELECT count(*) FROM profile_data").fetchone()[0]
print(f"프로파일 범위: {'전체' if PROFILE_ROWS is None else '앞부분'} {profile_count:,}행")


def profile_columns():
    expressions = []
    for name, dtype in root_types.items():
        col = qi(name)
        expressions.append(f"count(*) - count({col})")
        if dtype.id not in COMPLEX_TYPES:
            expressions.extend([
                f"approx_count_distinct({col})",
                f"CAST(min({col}) AS VARCHAR)",
                f"CAST(max({col}) AS VARCHAR)",
            ])
        else:
            expressions.extend(["NULL", "NULL", "NULL"])
    values = con.execute(f"SELECT {', '.join(expressions)} FROM profile_data").fetchone()
    records = []
    for i, (name, dtype) in enumerate(root_types.items()):
        nulls, distinct, minimum, maximum = values[i * 4:i * 4 + 4]
        records.append({
            "column": name, "type": str(dtype), "rows": profile_count,
            "nulls": nulls, "null_percent": 100 * nulls / profile_count if profile_count else None,
            "approx_distinct": distinct, "min": minimum, "max": maximum,
        })
    return pd.DataFrame(records)


profile = profile_columns()
display(profile)

## 4. 숫자 통계와 값 빈도

둘 다 위 `profile_data` 범위를 사용합니다. 숫자 통계는 NULL, NaN, 양/음의 무한대를 제외한 유한값 기준입니다.
`non_finite`는 NULL을 제외한 NaN/무한대의 수입니다. 기본값은 최상위 숫자 컬럼 전체입니다.
STRUCT 안의 숫자도 `NUMERIC_COLUMNS = [("상위", "숫자필드")]`로 지정할 수 있습니다.

빈도표는 기본적으로 첫 문자열/불리언 컬럼을 선택하며, 없으면 첫 일반 컬럼을 선택합니다.
NULL도 하나의 그룹으로 집계합니다. 중첩 컬럼은 별도 탐색 셀을 사용하세요.

In [ ]:
NUMERIC_COLUMNS = None  # None: 최상위 숫자 컬럼 전체, []: 생략
FREQUENCY_COLUMN = None  # None: 자동 선택, 또는 컬럼명/STRUCT 필드 경로
TOP_N = 20


def numeric_summary(columns):
    rows = []
    for field in columns:
        path = field_path(field)
        if field_types[path].id not in NUMERIC_TYPES:
            raise ValueError(f"숫자 필드가 아닙니다: {path}")
        col = field_sql(path)
        values = con.execute(f"""
            WITH values_to_check AS (SELECT {col} AS x FROM profile_data)
            SELECT count(x), count(*) FILTER (WHERE x IS NOT NULL AND NOT isfinite(x::DOUBLE)),
                   min(x) FILTER (WHERE isfinite(x::DOUBLE)),
                   max(x) FILTER (WHERE isfinite(x::DOUBLE)),
                   avg(x) FILTER (WHERE isfinite(x::DOUBLE)),
                   stddev_pop(x) FILTER (WHERE isfinite(x::DOUBLE)),
                   quantile_cont(x, [0.25, 0.5, 0.75]) FILTER (WHERE isfinite(x::DOUBLE))
            FROM values_to_check
        """).fetchone()
        count, non_finite, minimum, maximum, mean, std, quartiles = values
        q25, median, q75 = quartiles or [None, None, None]
        rows.append(dict(field=path, non_null=count, non_finite=non_finite,
                         min=minimum, max=maximum, mean=mean, std=std,
                         q25=q25, median=median, q75=q75))
    return pd.DataFrame(rows, columns=["field", "non_null", "non_finite", "min", "max", "mean", "std", "q25", "median", "q75"])


def value_counts(field, top_n=20):
    path = field_path(field)
    if field_types[path].id in COMPLEX_TYPES:
        raise ValueError("빈도표는 일반 필드를 선택하세요.")
    col = field_sql(path)
    return con.execute(f"""
        SELECT {col} AS value, count(*) AS count,
               100.0 * count(*) / nullif(?, 0) AS percent
        FROM profile_data GROUP BY {col}
        ORDER BY count DESC, value ASC NULLS LAST LIMIT ?
    """, [profile_count, positive_int(top_n, "TOP_N")]).fetchdf()


numeric_columns = ([name for name, dtype in root_types.items() if dtype.id in NUMERIC_TYPES]
                   if NUMERIC_COLUMNS is None else NUMERIC_COLUMNS)
display(numeric_summary(numeric_columns))
scalar_columns = [name for name, dtype in root_types.items() if dtype.id not in COMPLEX_TYPES]
preferred = [name for name in scalar_columns if root_types[name].id in {"varchar", "boolean"}]
frequency_column = FREQUENCY_COLUMN if FREQUENCY_COLUMN is not None else next(iter(preferred or scalar_columns), None)
if frequency_column is not None:
    print("빈도표:", frequency_column)
    display(value_counts(frequency_column, TOP_N))
else:
    print("빈도표를 만들 일반 컬럼이 없습니다.")

## 5. 조건 조회, 정렬, 페이지 이동

이 조회는 **전체 `data`**를 대상으로 합니다. `FILTERS`의 조건은 AND로 결합됩니다.
지원 연산자: `=`, `!=`, `<`, `<=`, `>`, `>=`, `in`, `not in`, `contains`, `is null`, `is not null`.

```python
SELECT_COLUMNS = ["name", "amount", ("address", "city")]
FILTERS = [("amount", ">=", 100), ("amount", "<", 200)]
FILTERS = [("date", ">=", "2026-01-01"), ("date", "<", "2026-02-01")]
FILTERS = [("category", "in", ["A", "B"]), ("name", "contains", "검색어")]
FILTERS = [("optional_column", "is null", None)]
ORDER_BY = [("date", "asc"), ("name", "desc")]
```

숫자/날짜 값은 실제 컬럼 타입으로 변환해 비교합니다. 날짜 종료 경계에 `<`를 쓰면 종료일은 제외됩니다.
`contains`는 와일드카드 없는 대소문자 구분 부분 문자열 검색입니다.
`in`/`not in`에 NULL을 넣지 말고 NULL 전용 연산자를 사용하세요. 일반 비교와 `not in`은 NULL 행을 제외합니다.

`OFFSET`으로 페이지를 이동할 수 있습니다. 일관된 페이지가 필요하면 고유 키까지 `ORDER_BY`에 지정하세요.
`LIMIT`은 반환 행 수를 제한하며, 스캔/정렬 비용까지 제한하지 않습니다.

In [ ]:
SELECT_COLUMNS = None  # None: 모든 최상위 컬럼
FILTERS = []
ORDER_BY = []  # 예: [("컬럼명", "asc")]
LIMIT = 1_000
OFFSET = 0


def build_query(selected_columns=None, filters=(), order_by=(), limit=1000, offset=0):
    selected = list(root_types) if selected_columns is None else selected_columns
    if not isinstance(selected, (list, tuple)) or not selected:
        raise ValueError("SELECT_COLUMNS는 None 또는 비어 있지 않은 목록이어야 합니다.")
    projection = ", ".join(
        f"{field_sql(field)} AS {qi('.'.join(field_path(field)))}" for field in selected
    )
    predicates, params = [], []
    for field, operator, value in filters:
        path = field_path(field)
        col, dtype = field_sql(path), field_types[path]
        op = operator.strip().lower()
        if op in {"is null", "is not null"}:
            if value is not None:
                raise ValueError("NULL 조건의 값은 None으로 지정하세요.")
            predicates.append(f"{col} {op.upper()}")
            continue
        if dtype.id in COMPLEX_TYPES:
            raise ValueError("중첩값 비교는 일반 하위 필드를 선택하거나 직접 SQL을 사용하세요.")
        cast = f"CAST(? AS {dtype})"
        if op in {"in", "not in"}:
            if not isinstance(value, (list, tuple)) or any(v is None for v in value):
                raise ValueError("in/not in에는 None이 없는 목록을 지정하세요.")
            if not value:
                predicates.append("FALSE" if op == "in" else "TRUE")
            else:
                predicates.append(f"{col} {op.upper()} ({', '.join([cast] * len(value))})")
                params.extend(value)
        elif op == "contains":
            if dtype.id != "varchar" or not isinstance(value, str):
                raise ValueError("contains는 문자열 컬럼과 문자열 검색어가 필요합니다.")
            predicates.append(f"contains({col}, ?)")
            params.append(value)
        elif op in {"=", "!=", "<", "<=", ">", ">="}:
            if value is None:
                raise ValueError("None 비교는 is null 또는 is not null을 사용하세요.")
            predicates.append(f"{col} {op} {cast}")
            params.append(value)
        else:
            raise ValueError(f"지원하지 않는 연산자: {operator}")
    ordering = []
    for field, direction in order_by:
        if direction.lower() not in {"asc", "desc"}:
            raise ValueError("정렬 방향은 asc 또는 desc입니다.")
        ordering.append(f"{field_sql(field)} {direction.upper()} NULLS LAST")
    sql = f"SELECT {projection} FROM data"
    if predicates:
        sql += " WHERE " + " AND ".join(predicates)
    if ordering:
        sql += " ORDER BY " + ", ".join(ordering)
    sql += " LIMIT ? OFFSET ?"
    params.extend([positive_int(limit, "LIMIT"), positive_int(offset, "OFFSET", allow_zero=True)])
    return sql, params


query_sql, query_parameters = build_query(SELECT_COLUMNS, FILTERS, ORDER_BY, LIMIT, OFFSET)
# 실패한 쿼리 뒤에 이전 결과를 실수로 내보내지 않도록 초기화합니다.
result = None
result_relations.pop("result", None)
result_relation = con.sql(query_sql, params=query_parameters)
result = result_relation.df()
result_relations["result"] = result_relation
print(f"반환 {len(result):,}행 / LIMIT {LIMIT:,} / OFFSET {OFFSET:,}")
if len(result) == LIMIT:
    print("행 제한에 도달했습니다. 조건에 맞는 행이 더 있을 수 있습니다.")
display(result)
print("SQL:", query_sql)
print("매개변수:", query_parameters)

## 6. 중첩값 탐색

`nested_fields`에서 경로를 골라 `NESTED_FIELD`에 지정하세요. 특정 컬럼명이나 식별자를 요구하지 않습니다.
STRUCT는 한 단계의 하위 필드를 컬럼으로 펼치고, LIST/ARRAY는 요소를 행으로 펼칩니다.
MAP은 key/value 행으로 바꿉니다. 요소가 STRUCT라면 한 단계 더 펼쳐 보여줍니다.

`PARENT_COLUMNS`에 원본 행을 식별할 컬럼을 지정하면 결과에 `parent.컬럼명`으로 보존합니다.
NULL/빈 목록은 펼친 결과에 행을 만들지 않습니다. `NESTED_PARENT_ROWS`는 펼치기 전에 읽을 부모 행 수,
`NESTED_LIMIT`은 펼친 뒤 반환할 행 수입니다. 한 행에 매우 큰 목록이 있으면 여전히 메모리가 필요합니다.
이 셀은 위 조건 조회와 독립적으로 원본 `data`의 앞부분을 탐색합니다.

In [ ]:
NESTED_FIELD = None  # 예: "items" 또는 ("payload", "items")
PARENT_COLUMNS = []
NESTED_PARENT_ROWS = 10
NESTED_LIMIT = 100

nested_fields = field_catalog[field_catalog["kind"].isin(["struct", "list", "array", "map"])]
display(nested_fields)


def inspect_nested(field, parent_columns=(), parent_rows=10, limit=100):
    path = field_path(field)
    dtype, col = field_types[path], field_sql(path)
    kind = dtype.id
    if kind not in {"struct", "list", "array", "map"}:
        raise ValueError("STRUCT/LIST/ARRAY/MAP 필드를 선택하세요.")
    parent_projection = [
        f"{field_sql(p)} AS {qi('parent.' + '.'.join(field_path(p)))}"
        for p in parent_columns
    ]
    value_expr = f"map_entries({col})" if kind == "map" else col
    if kind != "struct":
        value_expr = f"unnest({value_expr})"
    inner = ", ".join([*parent_projection, f"{value_expr} AS nested_value"])
    value_type = dtype if kind == "struct" else (
        dict(dtype.children)["child"] if kind in {"list", "array"} else None
    )
    if kind == "map":
        children = ["key", "value"]
    elif value_type.id == "struct":
        children = [name for name, _ in value_type.children]
    else:
        children = []
    outer = [qi('parent.' + '.'.join(field_path(p))) for p in parent_columns]
    if children:
        outer += [f"nested_value.{qi(name)} AS {qi('value.' + name)}" for name in children]
    else:
        outer += ['nested_value AS "value"']
    sql = f"""
        WITH parents AS (SELECT * FROM data LIMIT ?),
             expanded AS (SELECT {inner} FROM parents)
        SELECT {', '.join(outer)} FROM expanded LIMIT ?
    """
    return con.sql(sql, params=[positive_int(parent_rows, "NESTED_PARENT_ROWS"),
                               positive_int(limit, "NESTED_LIMIT")])


nested_result = None
result_relations.pop("nested_result", None)
if NESTED_FIELD is not None:
    nested_relation = inspect_nested(NESTED_FIELD, PARENT_COLUMNS, NESTED_PARENT_ROWS, NESTED_LIMIT)
    nested_result = nested_relation.df()
    result_relations["nested_result"] = nested_relation
    display(nested_result)
else:
    print("NESTED_FIELD를 지정하면 선택한 중첩값을 펼칩니다.")

## 7. 직접 SQL (선택)

원본 뷰는 `data`, 프로파일 범위는 `profile_data`입니다. 조합 조건, GROUP BY, 날짜별 집계,
여러 단계의 UNNEST 등은 여기서 작성하세요. 단일 SELECT/WITH 조회만 실행하며 반환 행 수를 제한합니다.
`?` 자리에 넣을 값은 `SQL_PARAMS`로 전달합니다. 일반 조건 조회 결과 `result`는 별도로 유지합니다.

```python
CUSTOM_SQL = 'SELECT "category", count(*) AS n FROM data GROUP BY "category" ORDER BY n DESC'
CUSTOM_SQL = 'SELECT * FROM data WHERE "amount" >= ?'
SQL_PARAMS = [100]
```

In [ ]:
CUSTOM_SQL = None
SQL_PARAMS = []
SQL_LIMIT = 1_000


def run_sql(sql, params=(), limit=1000):
    statements = con.extract_statements(sql)
    if len(statements) != 1 or statements[0].type != duckdb.StatementType.SELECT:
        raise ValueError("단일 SELECT/WITH 조회를 입력하세요.")
    return con.sql(sql, params=list(params)).limit(positive_int(limit, "SQL_LIMIT"))


sql_result = None
result_relations.pop("sql_result", None)
if CUSTOM_SQL:
    sql_relation = run_sql(CUSTOM_SQL, SQL_PARAMS, SQL_LIMIT)
    sql_result = sql_relation.df()
    result_relations["sql_result"] = sql_relation
    display(sql_result)

## 8. 조회 결과 저장 (선택)

`EXPORT_FORMAT`을 `"csv"` 또는 `"parquet"`으로 바꾸면 저장합니다.
`EXPORT_RESULT`는 `result`, `nested_result`, `sql_result` 중 선택합니다.
**현재 반환된 행만** 내보냅니다. LIMIT/OFFSET 이전의 전체 일치 데이터가 아닙니다.
CSV의 중첩값은 문자열 표현이므로 원본 타입 보존이 필요하면 Parquet를 사용하세요.
Parquet는 DataFrame을 다시 추론하지 않고 해당 DuckDB 조회를 다시 실행하여 원래 타입을 유지합니다.
두 실행 사이 원본 파일이 변경되면 저장 결과도 달라질 수 있습니다.
기존 파일은 덮어쓰지 않습니다. `exports/`는 Git에서 제외됩니다.

In [ ]:
EXPORT_FORMAT = None  # None, "csv", "parquet"
EXPORT_RESULT = "result"
EXPORT_STEM = "query_result"  # exports/ 아래 파일명 (확장자 제외)

if EXPORT_FORMAT is not None:
    if EXPORT_FORMAT not in {"csv", "parquet"}:
        raise ValueError("EXPORT_FORMAT은 None, csv, parquet 중 선택하세요.")
    frames = {"result": result, "nested_result": nested_result, "sql_result": sql_result}
    if EXPORT_RESULT not in frames or frames[EXPORT_RESULT] is None:
        raise ValueError("선택한 결과를 먼저 조회하세요.")
    if not EXPORT_STEM or Path(EXPORT_STEM).name != EXPORT_STEM or EXPORT_STEM in {".", ".."}:
        raise ValueError("EXPORT_STEM에는 폴더 경로 없이 파일명만 입력하세요.")
    export_path = PROJECT_ROOT / "exports" / f"{EXPORT_STEM}.{EXPORT_FORMAT}"
    export_path.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_FORMAT == "csv":
        frames[EXPORT_RESULT].to_csv(export_path, index=False, encoding="utf-8-sig", mode="x")
    else:
        # SQL 조회를 보존한 relation을 사용하므로 빈 결과/중첩/DECIMAL 타입도 유지합니다.
        export_relation = result_relations[EXPORT_RESULT]
        # 배타적 생성으로 기존 파일 보호. 실패한 새 파일만 정리합니다.
        with export_path.open("xb"):
            pass
        try:
            export_relation.write_parquet(str(export_path))
        except Exception:
            export_path.unlink(missing_ok=True)
            raise
    print(f"저장: {export_path}")